# Báo cáo Khoa học & Thuyết trình: Kiến trúc Governed RAG trong Cố vấn Học vụ

**Dự án**: Intelligent Student Advisor Platform (Capstone)  
**Chủ đề**: Mô hình RAG Đa tầng có Kiểm soát Hiệu lực Thời gian và Dẫn nguồn Chính xác (*Governed & Grounded RAG*)  
**Mục đích**: Sổ tay khoa học trực quan phục vụ báo cáo thuyết trình, bao gồm cơ sở lý thuyết, sơ đồ kiến trúc, bảng so sánh công nghệ và mã nguồn thực nghiệm trực tiếp.


## 1. Đặt vấn đề: Tại sao Naive RAG thất bại trong Cố vấn Học vụ?

Khi ứng dụng Mô hình Ngôn ngữ Lớn (LLM) vào tư vấn quy chế đào tạo đại học, hệ thống gặp phải **3 rủi ro nghiêm trọng**:

1. **Ảo giác thông tin (*Hallucination*)**: LLM tự suy đoán điều kiện tốt nghiệp, điểm rèn luyện, chuẩn ngoại ngữ khi tài liệu không đề cập.
2. **Xung đột hiệu lực thời gian (*Temporal Policy Shifts*)**: Quy chế đào tạo thay đổi theo từng năm/khóa (ví dụ: Quy chế 2021 khác 2024). RAG thông thường dễ lấy nhầm điều khoản của khóa cũ áp dụng cho sinh viên khóa mới.
3. **Thiếu khả năng truy vết pháp lý (*Auditability*)**: Sinh viên và cố vấn cần số liệu dẫn nguồn chính xác đến từng **Trang** và **Đoạn** của văn bản Nhà trường ban hành.


### Bảng so sánh các thế hệ kiến trúc RAG

| Tiêu chí | Naive RAG (RAG thông thường) | Advanced RAG | **Governed RAG (Dự án đề xuất)** |
|---|---|---|---|
| **Nguồn dữ liệu** | Văn bản thuần (TXT, MD) | PDF văn bản điện tử | **Đa phương thức**: TXT, MD, DOCX, PDF văn bản, **PDF scan & Ảnh scan (PNG, JPG) qua Local OCR** |
| **Bảo vệ tài nguyên & DoS** | Không giới hạn | Giới hạn dung lượng file | **Rào chắn 4 tầng**: Max 20 trang, 10M pixels, 10s timeout, 100K ký tự |
| **Kiểm soát thời gian hiệu lực** | Không có (tìm kiếm toàn bộ) | Lọc metadata tĩnh | **Gated Temporal Validation**: $valid\_from \le as\_of < valid\_until$, loại bỏ 100% văn bản hết hạn |
| **Chiến lược phân đoạn (*Chunking*)** | Cố định số ký tự có chồng lấn | Tách theo câu đơn | **Structure-Aware Chunking**: Giữ nguyên ranh giới đoạn, trích xuất `## Trang {N}` $\rightarrow$ Dẫn nguồn `Trang X, Đoạn Y` |
| **Phương thức truy xuất** | Chỉ Dense Embedding | Hybrid (Dense + Sparse) | **Truy xuất lai đa tầng**: Okapi TF-IDF / BM25+ và Dense Multilingual-E5 (768 chiều, pinned commit) |
| **Xác thực chống ảo giác** | Chỉ dựa vào System Prompt | Đo khoảng cách Cosine | **Citation Verification Engine**: Kiểm duyệt từng claim, bắt buộc dẫn `chunk_id` có thật trong context; nếu bịa citation $\rightarrow$ **Abstain** |
| **Chế độ bảo mật** | Gọi Cloud API | Phụ thuộc OpenAI | **100% On-Premise / Local Offline**: Không rò rỉ dữ liệu quy chế hay thông tin sinh viên ra bên ngoài |


## 2. Sơ đồ Kiến trúc Tổng thể (Architecture Pipeline)

```
+-----------------------------------------------------------------------------------------+
|                                  1. INGESTION PIPELINE                                  |
|  Tài liệu (PDF, DOCX, Scan) --> Local OCR (Tesseract-Vie) --> Structure-Aware Chunker  |
|  --> PostgreSQL (Băm SHA256, app.chunks) & ChromaDB (Multilingual-E5 Vectors)           |
+-----------------------------------------------------------------------------------------+
                                            |
                                            v
+-----------------------------------------------------------------------------------------+
|                           2. GATED HYBRID RETRIEVAL PIPELINE                            |
|  Câu hỏi sinh viên + as_of (ngày truy vấn)                                              |
|  --> [BỘ LỌC THỜI GIAN]: valid_from <= as_of < valid_until                              |
|  --> [TRUY XUẤT LAI]: Lexical BM25 (từ khóa) + Dense E5 (ngữ nghĩa cosine)              |
|  --> Top-K Evidence Chunks (kèm mã hash và vị trí Trang X, Đoạn Y)                      |
+-----------------------------------------------------------------------------------------+
                                            |
                                            v
+-----------------------------------------------------------------------------------------+
|                        3. GROUNDED GENERATION & VERIFICATION                            |
|  Đóng gói Prompt Ngữ cảnh (Chỉ cho phép dùng Evidence) --> LLM Inference Engine         |
|  --> [CITATION VERIFIER]: Bắt buộc trích dẫn [citation:chunk_id]                        |
|      + Hợp lệ: Trả lời sinh viên kèm dẫn nguồn Trang X, Đoạn Y                          |
|      + Bịa đặt / Thiếu bằng chứng: Từ chối trả lời an toàn (insufficient_evidence)     |
+-----------------------------------------------------------------------------------------+
```


## 3. Thực nghiệm 1: Phân mảnh Ngữ nghĩa & Nhận thức Trang (Structure-Aware Chunking)

Module `advisor_core.rag.chunk_markdown` tự động phân tích tiêu đề `## Trang {N}` để tạo nhãn dẫn nguồn chi tiết `Trang X, Đoạn Y` và băm SHA256 định danh bất biến.


In [ ]:
import hashlib
import pandas as pd
from advisor_core.rag import chunk_markdown

# Văn bản quy chế mô phỏng sau khi qua bộ trích xuất OCR
sample_policy = """
## Trang 1

Điều 14: Điều kiện xét tốt nghiệp đại học hệ chính quy theo hệ thống tín chỉ.
Sinh viên được công nhận tốt nghiệp khi tích lũy đủ số tín chỉ quy định của chương trình đào tạo.

Điểm trung bình tích lũy (CPA) toàn khóa học phải đạt từ 2.00 trở lên theo thang điểm 4.

## Trang 2

Điều 15: Chuẩn đầu ra ngoại ngữ và công nghệ thông tin.
Sinh viên ngành Công nghệ thông tin phải đạt chứng chỉ tiếng Anh chuẩn B1 hoặc TOEIC 450 trở lên.

Hoàn thành chứng chỉ Giáo dục quốc phòng và Giáo dục thể chất theo quy định của Bộ Giáo dục và Đào tạo.
"""

chunks = chunk_markdown(
    sample_policy,
    document_id="doc-quy-che-2026",
    version_id="ver-1",
    scope="academic_policy",
    source="Quy chế Đào tạo Tín chỉ Số 123/QĐ-ĐH",
    valid_from="2026-09-01"
)

df_chunks = pd.DataFrame([{
    "Vị trí dẫn nguồn": c["section"],
    "Mã Chunk ID (8 ký tự đầu)": c["chunk_id"][:8] + "...",
    "Độ dài ký tự": len(c["text"]),
    "Nội dung đoạn": c["text"].replace("\n", " ")
} for c in chunks if not c["text"].startswith("##")])

print(f"Tổng số đoạn trích hợp lệ được tạo: {len(df_chunks)}")
df_chunks


ModuleNotFoundError: No module named 'advisor_core'

: 

## 4. Thực nghiệm 2: Cơ chế Lọc Ngữ cảnh theo Thời gian (Temporal & Scope Gating)

Công thức toán học của bộ lọc:
$$\text{Applicable}(C, \text{scope}, T) = \{ c \in C \mid c.\text{scope} = \text{scope} \land c.\text{valid\_from} \le T < c.\text{valid\_until} \land c.\text{status} = \text{'approved\_demo'} \}$$

Đoạn mã sau kiểm chứng: Sinh viên hỏi tại năm 2026 sẽ **không bao giờ** bị lấy nhầm văn bản cũ đã hết hạn từ năm 2024.


In [2]:
from advisor_core.rag import applicable

corpus = [
    {
        "chunk_id": "c-old-2021",
        "scope": "academic_policy",
        "status": "approved_demo",
        "valid_from": "2021-09-01",
        "valid_until": "2024-08-31",  # ĐÃ HẾT HIỆU LỰC
        "text": "Quy chế 2021: Sinh viên chỉ cần chứng chỉ tiếng Anh A2 để tốt nghiệp."
    },
    {
        "chunk_id": "c-new-2024",
        "scope": "academic_policy",
        "status": "approved_demo",
        "valid_from": "2024-09-01",
        "valid_until": "9999-12-31",  # ĐANG ÁP DỤNG
        "text": "Quy chế 2024: Sinh viên bắt buộc phải có chứng chỉ B1 (TOEIC 450) để tốt nghiệp."
    }
]

query_date = "2026-09-06"
active_chunks = applicable(corpus, scope="academic_policy", as_of=query_date)

print(f"=== KẾT QUẢ LỌC HIỆU LỰC TẠI NGÀY {query_date} ===")
print(f"Số văn bản trong kho: {len(corpus)}")
print(f"Số văn bản thỏa mãn hiệu lực: {len(active_chunks)}")
for ac in active_chunks:
    print(f"- [HỢP LỆ]: {ac['chunk_id']} (Từ {ac['valid_from']} đến {ac['valid_until']})")
    print(f"  Nội dung: {ac['text']}")


## 5. Thực nghiệm 3: Truy xuất Từ khóa Lai (Lexical BM25+ Search)

Triển khai thuật toán Okapi BM25+ trên không gian token Unicode chuẩn hóa tiếng Việt:
$$\text{Score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{avgdl}\right)}$$


In [3]:
from advisor_core.rag import lexical_search

test_corpus = [
    {
        "chunk_id": "c-01", "scope": "academic_policy", "status": "approved_demo",
        "valid_from": "2026-01-01", "valid_until": "9999-12-31", "section": "Trang 1, Đoạn 1",
        "text": "Quy định học lại: Sinh viên có điểm tổng kết dưới 4.0 phải đăng ký học lại học phần này."
    },
    {
        "chunk_id": "c-02", "scope": "academic_policy", "status": "approved_demo",
        "valid_from": "2026-01-01", "valid_until": "9999-12-31", "section": "Trang 1, Đoạn 2",
        "text": "Điều kiện xét tốt nghiệp: Sinh viên phải tích lũy tối thiểu 126 tín chỉ và CPA từ 2.00 trở lên."
    },
    {
        "chunk_id": "c-03", "scope": "academic_policy", "status": "approved_demo",
        "valid_from": "2026-01-01", "valid_until": "9999-12-31", "section": "Trang 2, Đoạn 1",
        "text": "Chính sách học bổng khuyến khích: Sinh viên có GPA từ 3.60 và điểm rèn luyện xuất sắc được xét học bổng."
    }
]

query = "Cần bao nhiêu tín chỉ và điểm CPA để được xét tốt nghiệp?"
results = lexical_search(test_corpus, query=query, scope="academic_policy", as_of="2026-09-06", k=2)

print(f"Câu hỏi: '{query}'")
print("=== KẾT QUẢ TRUY XUẤT XẾP HẠNG (TOP-K) ===")
for rank, r in enumerate(results, start=1):
    chunk = r["chunk"]
    print(f"{rank}. [Điểm BM25: {round(r['score'], 4)}] {chunk['section']} (ID: {chunk['chunk_id']})")
    print(f"   Nội dung: {chunk['text']}\n")


## 6. Thực nghiệm 4: Bộ Kiểm duyệt Chống Ảo giác (Citation Verification Engine)

Mỗi nhận định trong câu trả lời bắt buộc phải kèm một trích dẫn `chunk_id` có thật trong danh sách kết quả truy xuất.
- **Trường hợp A (Hợp lệ)**: LLM trích dẫn đúng `chunk_id` có trong context $\rightarrow$ Trả về câu trả lời hoàn chỉnh.
- **Trường hợp B (Ảo giác / Bịa citation)**: LLM trích dẫn `chunk_id` không tồn tại $\rightarrow$ Hệ thống tự động kích hoạt **Abstain**, chuyển sang trạng thái `insufficient_evidence`.


In [4]:
def verify_claims_and_citations(generated_claims, retrieved_chunk_ids):
    valid_chunks = set(retrieved_chunk_ids)
    all_valid = True
    checked_claims = []
    for claim in generated_claims:
        cited = set(claim.get("citations", []))
        # Kiểm tra xem citation có nằm trong tập bằng chứng không
        is_grounded = cited.issubset(valid_chunks) and len(cited) > 0
        if not is_grounded:
            all_valid = False
        checked_claims.append({
            "mệnh_đề": claim["text"],
            "citations": list(cited),
            "xác_thực": "HỢP LỆ" if is_grounded else "ẢO GIÁC (BỊA NGUỒN)"
        })
    
    status = "answered" if all_valid else "insufficient_evidence (abstain)"
    return status, checked_claims

# Giả sử các chunk đã truy xuất được
retrieved_ids = ["c-02"]

# Case 1: LLM sinh đúng căn cứ
claims_correct = [{
    "text": "Sinh viên phải tích lũy tối thiểu 126 tín chỉ và CPA từ 2.00 để tốt nghiệp.",
    "citations": ["c-02"]
}]

# Case 2: LLM bịa nguồn không tồn tại trong context
claims_hallucinated = [{
    "text": "Sinh viên được miễn học phí nếu thuộc diện gia đình chính sách.",
    "citations": ["c-fake-999"]  # Nguồn bịa
}]

status_1, report_1 = verify_claims_and_citations(claims_correct, retrieved_ids)
status_2, report_2 = verify_claims_and_citations(claims_hallucinated, retrieved_ids)

print("=== TEST CASE 1: CÂU TRẢ LỜI CÓ DẪN NGUỒN CHÍNH XÁC ===")
print(f"Trạng thái hệ thống: {status_1}")
print(report_1)

print("\n=== TEST CASE 2: CÂU TRẢ LỜI CÓ NGUỒN ẢO GIÁC ===")
print(f"Trạng thái hệ thống: {status_2} --> Hệ thống từ chối khẳng định!")
print(report_2)


## 7. Bảng Thông số Kỹ thuật Hệ thống (System Specifications)

| Thành phần | Công nghệ / Tham số kỹ thuật | Ghi chú thiết kế |
|---|---|---|
| **Mô hình Dense Embedding** | `intfloat/multilingual-e5-base` (768 chiều) | Pinned Git commit SHA256, prefix `query: ` và `passage: ` |
| **Vector Database** | ChromaDB PersistentClient | Khoảng cách Cosine (`hnsw:space: cosine`) |
| **Cơ sở dữ liệu Toàn văn** | PostgreSQL 17 (`app.chunks`, `app.documents`) | Băm SHA256 toàn vẹn nội dung |
| **Kích thước đoạn (Chunk)** | 1.400 – 1.600 ký tự | Tách theo ranh giới đoạn văn tự nhiên |
| **Công cụ OCR** | Tesseract 5.x (`vie` + `eng`) + `pdf2image` | Render 150 DPI, khử nhiễu, phân tách trang scan |
| **Giới hạn an toàn OCR** | $\le$ 20 trang, $\le$ 10.000.000 pixels, timeout 10s/trang | Chống tấn công Decompression Bomb / DoS |
| **Số lượng ứng viên (Top-K)**| $K = 5$ đoạn trích phù hợp nhất | Cân bằng ngữ cảnh và giới hạn context window |
| **Định dạng hiển thị trích dẫn**| `Trang X, Đoạn Y` | Cho phép sinh viên tra cứu trực tiếp văn bản giấy |


## 8. Kết luận & 4 Điểm nhấn Bảo vệ Đồ án (Key Takeaways for Defense)

1. **Tính Thực tiễn & Địa phương hóa (Local & Feasible)**: Hệ thống xử lý trực tiếp các quyết định quy chế dạng scan/ảnh của Nhà trường qua pipeline Local OCR tiếng Việt, hoàn toàn chạy offline trên Docker.
2. **An toàn Pháp lý theo Thời gian (Temporal Safety)**: Cơ chế Temporal Gating đảm bảo sinh viên luôn được tư vấn đúng văn bản đang có hiệu lực tại thời điểm tra cứu.
3. **Triệt tiêu Ảo giác (Zero-Hallucination Guard)**: Tự động chuyển về trạng thái Abstain nếu không tìm thấy bằng chứng pháp lý xác thực.
4. **Khả năng giải trình & Kiểm toán (Granular Auditability)**: Dẫn nguồn rõ ràng đến từng trang và đoạn văn bản, đáp ứng tiêu chuẩn khắt khe của môi trường giáo dục đại học.
